# Chapter 15 — Scan Operations

Part 6: Reductions, scans, and combined recipes.

**Ladder.**

- **Method 01:** naive Python scan loop.
- **Method 02:** `tl.cumsum` / `tl.associative_scan` kernels.
- **Method 03:** applied scans — position ids, segmented sums, stream compaction, first-order recurrence.

**Thread.**

- **Recurring idea:** a scan is a reduction that keeps every prefix; the operator is associative and the output shape never changes.
- **Chain:** Chapter 14 reduction → 15.1 prefix sum → 15.2 cumulative sum → 15.3 `tl.associative_scan` → 15.4 inclusive → 15.5 exclusive → 15.6 parallel scan → 15.7 reduction vs scan → 15.8 applications.
- **Shape thread:** a scan preserves every axis; a reduction removes the reduced axis.
- **Offset thread:** the exclusive scan is the offset generator behind positions, compaction, and segments.

**Study tip.**

- Take $x=(3,1,4,1,5)$.
- Inclusive keeps the current element: $(3,\ 3+1,\ 3+1+4,\ 3+1+4+1,\ 3+1+4+1+5)=(3,4,8,9,14)$.
- Exclusive drops it and starts at the identity $0$: $(0,\ 3,\ 3+1,\ 3+1+4,\ 3+1+4+1)=(0,3,4,8,9)$.
- Ten numbers, one pattern: $e_i=y_{i-1}$ with $y_{-1}=0$. The last inclusive value is the reduction $14$; the last exclusive value is $14-5=9$.
- Reproduce those numbers and every kernel below is a scheduling detail.

**Learning objectives**

- Read a scan as a reduction that keeps every prefix, and state the difference in **output shape**.
- Choose inclusive or exclusive, and convert one into the other with a shift and the identity element.
- Write `tl.cumsum` and `tl.associative_scan` kernels; state the associativity requirement on the combine function.
- Implement Hillis–Steele doubling by hand and count its work and depth.
- Apply a scan to position ids, segmented sums, stream compaction, and a first-order recurrence.
- Explain why a scan is not a reduction with a loop around it.

**Estimated time:** 70–85 min · **Difficulty:** Intermediate

**Prerequisites**

- Chapter 14: `tl.sum`, `tl.max`, axis reductions, and the output-shape rule for reductions.
- Chapter 5: masked loads and stores, pointer offsets, and `other=`.
- Comfort with the summation notation $\sum_{j=0}^{i}$ and base-2 logarithms.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch14_reductions/14_complete.ipynb) · [Next →](../ch16_combining_ops/16_complete.ipynb)


## Concept diagrams

### Reduction vs scan

- **Reduction.** Collapse the axis to one value.
- **Scan.** Keep the running value at every position.

```text
x:            3     1     4     1     5
              |     |     |     |     |
reduction:    +-----+-----+-----+-----+----> 14            (one output)

scan:         +-->  3
                    +-->  4
                          +-->  8
                                +-->  9
                                      +--> 14              (n outputs)
```

- **Output count.** For a length-$N$ axis, a reduction has **one** output and a scan has **$N$**.

### Inclusive vs exclusive

- **Inclusive.** Include the current element.
- **Exclusive.** Drop it and start at the identity.

| $i$ | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| $x_i$ | 3 | 1 | 4 | 1 | 5 |
| inclusive $y_i$ | 3 | 4 | 8 | 9 | 14 |
| exclusive $e_i$ | 0 | 3 | 4 | 8 | 9 |

- **Shift.** The exclusive row is the inclusive row shifted right by one, with the identity in slot zero:

```text
inclusive:   3   4   8   9  14
              \   \   \   \
exclusive: 0   3   4   8   9
```

### The Hillis–Steele doubling tree

- **Rule.** Each round doubles the distance a value travels.
- **Rounds.** For $N=8$, three rounds suffice.

```text
round 0 (input): 1   2   3   4   5   6   7   8
round 1 (d=1):   1   3   5   7   9  11  13  15
round 2 (d=2):   1   3   6  10  14  18  22  26
round 3 (d=4):   1   3   6  10  15  21  28  36

lane i in round k reads lane i - 2^(k-1) from round k-1.
```

### Work and efficiency

| Algorithm | Work $W$ | Depth $D$ | Parallelism $W/D$ | Note |
|---|---:|---:|---:|---|
| sequential loop | $N-1$ | $N-1$ | $1$ | one lane does every add |
| Hillis–Steele | $\approx N\log_2 N$ | $\lceil\log_2 N\rceil$ | $\approx N$ | more work, less depth |
| Blelloch | $2(N-1)$ | $2\lceil\log_2 N\rceil$ | $\approx N/2$ | work-efficient, two phases |

- **Cost.** The scan is not free: it trades work for depth. For large $N$ it is also memory-bound, with roughly $2Nb$ bytes of traffic for $N$ elements of $b$ bytes.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, require_gpu, run_if_gpu, next_pow2

print("device:", DEVICE, "| GPU:", HAS_GPU)


## Part I — Mathematics and concepts

Write $[N]=\{0,\dots,N-1\}$. A scan replaces each element by the combination of everything up to it; the operator must be **associative**, and for the sum scan it is $+$ with identity $0$.

### 15.1 Prefix sum

- **Connection.** Builds on Chapter 14: a reduction (`tl.sum`) collapses an axis to one value, while a scan keeps the running value at every position.
- **Rule.** Carry one running value along the axis, combining each new element into it and emitting the carry at every stop.
- **Formula and shape.** $y_i=\bigoplus_{j=0}^{i}x_j$, which is $y_i=\sum_{j=0}^{i}x_j$ when $\oplus=+$; $y_0=x_0$ and $y_{N-1}=\sum_{j=0}^{N-1}x_j$. A scan preserves shape, $\operatorname{shape}(y)=\operatorname{shape}(x)$: $(N,)$ stays $(N,)$, and $(R,C)$ stays $(R,C)$ for `axis=0` (down columns) or `axis=1` (across rows). A reduction differs: `tl.sum(A, axis=0)` returns $(C,)$.
- **Example.** On $x=(3,1,4,1,5)$: $y=(3,4,8,9,14)$.
- **Where.** Prefix sums turn per-element quantities into offsets: row offsets in a ragged batch, bucket starts in a histogram, cumulative probability in sampling, and the integral image in vision. They are also the forward pass of a linear recurrence with coefficient one.
- **Triton.** `tl.cumsum(x, axis=0)` is the inclusive sum scan. The generic form is `tl.associative_scan(x, axis=0, combine_fn=_add)` with a `@triton.jit` combine function. Load the masked tail with `other=0.0`: the identity is what makes the padding harmless.
- **Failure.** Using `tl.sum` when every prefix is needed, or scanning with `axis` pointing at the wrong dimension. A scan along the batch axis is almost never what a row-wise normalization wants.
- **Bridge.** Section 15.2 gives this operation its library name and its 2-D row-wise form.


### 15.2 Cumulative sum

- **Connection.** Builds on 15.1: cumulative sum is the same inclusive prefix sum under its library name, viewed row by row.
- **Rule.** Scan each row independently from left to right; the library name is `cumsum`.
- **Formula and shape.** Along axis 1, $C_{r,c}=\sum_{k=0}^{c}A_{r,k}$. The shape is preserved, $(R,C)\to(R,C)$, for either axis. Axis 1 is the common case because each row is an independent sequence; a reduction along axis 1 would return $(R,)$ instead.
- **Example.** Row $(1,2,3)$ maps to $(1,3,6)$: $1,\ 1+2=3,\ 3+3=6$. Row $(4,5,6)$ maps to $(4,9,15)$: $4,\ 4+5=9,\ 9+6=15$.
- **Where.** Running totals over time, cumulative rewards, per-row normalizers, and the "prefix" half of a prefix-sum + difference pair. In a transformer, a cumulative sum over a mask gives position ids.
- **Triton.** `tl.cumsum(x, axis=1)` on a 2-D tile. The axis is a compile-time constant; a 2-D tile must be loaded as a block, not as a Python list of rows.
- **Failure.** Flattening a matrix before the scan. `x.reshape(-1)` scans across row boundaries and produces a different answer; keep the axis.
- **Bridge.** Section 15.3 replaces $+$ with any associative operator.


### 15.3 `tl.associative_scan()`

- **Connection.** Builds on 15.2: `tl.cumsum` is the $+$ specialization of this general primitive.
- **Rule.** Combine adjacent values with an associative operator $\oplus$ that has an identity $e$; the tree may re-parenthesize freely.
- **Formula and shape.** $y_i=x_0\oplus x_1\oplus\cdots\oplus x_i$, licensed by $(a\oplus b)\oplus c=a\oplus(b\oplus c)$. The scan preserves the shape along `axis` and the dtype of the state. For a tuple of $k$ tensors, `combine_fn` takes $2k$ scalar arguments, returns $k$ values, and every state tensor must have the same shape.
- **Example.** On $x=(3,1,4,1,5)$, $\oplus=+$ gives $(3,4,8,9,14)$ and $\oplus=\max$ gives $(3,3,4,4,5)$. The max scan is a running maximum; it is valid because $\max$ is associative, with identity $-\infty$.
- **Where.** Any monoid works: sum, product, min, max, gcd, bitwise and/or, $(\max,\operatorname{argmax})$ for a running argmax, affine maps for recurrences, and `(head_flag, value)` for segmented scans.
- **Triton.** `y = tl.associative_scan(x, axis=0, combine_fn=_add)` with a `@triton.jit` `_add`. The combine function must be jitted; Triton lowers the scan to a tree, so the combine order is not the left-to-right order of the formula.
- **Failure.** A non-associative operator such as subtraction: the left fold is $-8$ and the balanced fold is $2$ on the spine. Floating-point addition is only approximately associative, so a scan can differ from a loop in the last bits.
- **Bridge.** Section 15.4 fixes the alignment used by these examples: the inclusive convention.


### 15.4 Inclusive scan

- **Connection.** Builds on 15.3's generic scan; inclusive is the default alignment, with the current element inside the prefix.
- **Rule.** Include $x_i$ in $y_i$; the first output is the first input.
- **Formula and shape.** $y_i=x_0\oplus\cdots\oplus x_i$ and $y_0=x_0$. The shape is preserved, $(N,)\to(N,)$ and $(R,C)\to(R,C)$ along the chosen axis. Output $i$ depends only on $x_0,\dots,x_i$; changing $x_j$ with $j\gt i$ cannot change $y_i$.
- **Example.** $x=(3,1,4,1,5)\mapsto y=(3,4,8,9,14)$; the last value is the total reduction.
- **Where.** The default scan in NumPy, PyTorch, and Triton: running totals, cumulative maxima, and any "value so far" quantity.
- **Triton.** `tl.cumsum(x, axis=0)` or `tl.associative_scan(x, axis=0, combine_fn=_add)`.
- **Failure.** An off-by-one in a downstream formula that expected the exclusive convention. The first element is $x_0$, not the identity, and the output is one element longer than the number of gaps.
- **Bridge.** Section 15.5 drops the current element and starts from the identity.


### 15.5 Exclusive scan

- **Connection.** Builds on 15.4: the exclusive scan is the inclusive scan with the prefix shifted one slot to the right.
- **Rule.** Exclude $x_i$ from $y_i$; slot zero holds the identity.
- **Formula and shape.** $y_i=x_0\oplus\cdots\oplus x_{i-1}$ and $y_0=e$. For addition $e=0$; for multiplication $e=1$; for max $e=-\infty$. The output shape equals the input shape; only the alignment changes.
- **Example.** $x=(3,1,4,1,5)\mapsto y=(0,3,4,8,9)$. The shift relation is $e_i=y_{i-1}$ with $y_{-1}=e$. For an invertible operator such as addition there is a second route: $e_i=y_i-x_i=(0,3,4,8,9)$.
- **Where.** Exclusive scans produce **offsets**, not totals. Stream compaction writes item $i$ to slot $e_i$, so the first kept item lands at slot $0$. The same offsets place nonzeros, bucket boundaries, and variable-length segments.
- **Triton.** There is no `exclusive=True` flag. Shift the input and scan: `shifted = tl.load(x_ptr + offs - 1, mask=(offs > 0) & mask, other=0.0)` then `exclusive = tl.cumsum(shifted, axis=0)`. The masked lane at `offs == 0` receives the identity from `other`.
- **Failure.** Computing `inclusive - x` for a non-invertible operator. It works for sum but not for max, min, or a general monoid; use the shift, which works for every associative operator with an identity.
- **Bridge.** Section 15.6 asks how to compute all of these prefixes in parallel.


### 15.6 Parallel scan

- **Connection.** Builds on 15.5: with both alignments defined, the remaining question is scheduling — a serial loop or a doubling tree.
- **Rule.** Hillis–Steele doubles the distance each round. With $y_i^{(0)}=x_i$ and $d=1,2,4,\dots$, set $y_i^{(d)}=y_i^{(d/2)}\oplus y_{i-d}^{(d/2)}$ for $i\ge d$, otherwise keep $y_i^{(d/2)}$.
- **Formula and shape.** After $\lceil\log_2 N\rceil$ rounds, $y_i=x_0\oplus\cdots\oplus x_i$. The shape is preserved and the round count depends on $N$, not on the values.
- **Example.** For $x=(1,2,3,4,5,6,7,8)$:

| round | $d$ | values after the round |
|---|---:|---|
| 0 | — | 1 2 3 4 5 6 7 8 |
| 1 | 1 | 1 3 5 7 9 11 13 15 |
| 2 | 2 | 1 3 6 10 14 18 22 26 |
| 3 | 4 | 1 3 6 10 15 21 28 36 |

Round 2, lane 4: $9+5=14$. Round 3, lane 7: $26+10=36$.
- **Where.** A GPU has thousands of lanes; a left-to-right loop uses one. The tree exposes $N$ independent adds per round, at the price of $N\log_2 N$ combines instead of $N-1$.
- **Triton.** `tl.associative_scan` already lowers to a tree. The manual kernel uses `tl.gather` and `tl.where` to make each step explicit: `shifted = tl.gather(acc, tl.maximum(offs - d, 0), axis=0)` then `acc = acc + tl.where(offs >= d, shifted, 0.0)`.
- **Failure.** Assuming that one program per block produces a global prefix. Within a block the tree is correct; across blocks, block $p$ must add the totals of blocks $0,\dots,p-1$. That carry is a second, tiny scan (an exercise).
- **Bridge.** Section 15.7 separates the full scan from the reduction that only keeps the last prefix.


### 15.7 Reduction vs scan

- **Connection.** Builds on 15.6: the tree computes every prefix, of which a reduction keeps only the last.
- **Rule.** A reduction combines the whole axis to one value; a scan emits the running combination at every position.
- **Formula and shape.** Reduce: $r=\bigoplus_{j=0}^{N-1}x_j$. Scan: $y_i=\bigoplus_{j=0}^{i}x_j$. A reduction removes the reduced axis: $(R,C)$ with `axis=0` becomes $(C,)$, or $(1,C)$ with `keep_dims=True`. A scan preserves it: $(R,C)\to(R,C)$. The last inclusive value is the reduction.
- **Example.** On $x=(3,1,4,1,5)$: $r=14$, $y=(3,4,8,9,14)$, and $y_{N-1}=r$.
- **Where.** Use a scan only when every prefix is consumed. A softmax needs one denominator, so it uses a reduction; a position-id or compaction kernel needs every offset, so it uses a scan. A scan immediately indexed at `[-1]` is a reduction with extra work.
- **Triton.** `tl.sum(x, axis=0)` versus `tl.cumsum(x, axis=0)`. One kernel can produce both: store the prefix tile, then store `tl.sum(x, axis=0)` in a one-element buffer.
- **Failure.** Using a reduction when the application needs offsets (a correctness bug), or using a scan when it needs only a total (a performance bug).
- **Bridge.** Section 15.8 collects the five application patterns built from these tools.


### 15.8 Applications of scan

- **Connection.** Builds on 15.1–15.7: each application is one of the scans above plus a mask, a scatter, or a tuple state.
- **Rule.** Five patterns recur.

| Application | Scan | Formula |
|---|---|---|
| running total | inclusive sum | $y_i=\sum_{j\le i}x_j$ |
| position ids | exclusive sum of ones | $p_i=\sum_{j\lt i}1=i$ |
| stream compaction | exclusive sum of a keep mask | $\operatorname{pos}_i=\sum_{j\lt i}k_j$ |
| segmented sum | tuple scan with a head flag | $(h,v)\oplus(h',v')=(h\lor h',\ v'\ \text{if }h'\text{ else }v+v')$ |
| first-order recurrence | affine tuple scan | $(a,b)\oplus(a',b')=(aa',\ a'b+b')$ |

- **Formula and shape.** Every application preserves the input shape. Segmented scans carry a second state tensor (the head flag) of the same shape. Compaction changes the *storage* size — it writes only the kept items — but the scan that computes the offsets still has the input shape.
- **Example.** With $x=(1,2,3,4,5,6)$ and head flags $h=(1,0,0,1,0,0)$, the segmented scan is $(1,3,6,\ 4,9,15)$, so the per-segment positions are $(0,1,2,0,1,2)$ after subtracting one. For $y_i=a_i y_{i-1}+b_i$ with $a=(1,2,3,4,5)$ and $b=(1,1,1,1,1)$, $y=(1,3,10,41,206)$. Check lane 3: $4\cdot10+1=41$.
- **Where.** Ragged batching (positions and segment sums), filtering (compaction), deduplication, run-length decoding, IIR filters, and dynamic programming recurrences. A scan turns a serial dependency into a logarithmic-depth computation.
- **Triton.** `tl.cumsum` for the first three; `tl.associative_scan` with a tuple and a `@triton.jit` combine function for the last two.
- **Failure.** Ignoring segment boundaries: a plain cumsum over a batch of concatenated sequences counts across the boundary. Compaction is stable only if the scatter uses the exclusive offsets in input order; a different order permutes equal keys.
- **Bridge.** The five patterns are the bridge from the theory to the applied kernels in Part IV.

**Where this leads.** Part II turns the definitions into a naive loop, Part III maps them to Triton kernels, and Part IV applies them to positions, compaction, and recurrences. Chapter 16 then composes reductions and scans into softmax, log-sum-exp, and normalization.


## Chapter map

| § | Topic | Formula | Shape/associativity rule | Kernel/demo |
|---|---|---|---|---|
| **15.1** | Prefix sum | $y_i=\sum_{j=0}^{i}x_j$ | $\operatorname{shape}(y)=\operatorname{shape}(x)$ | `inclusive_scan_kernel` |
| **15.2** | Cumulative sum | $C_{r,c}=\sum_{k=0}^{c}A_{r,k}$ | axis preserved: $(R,C)\to(R,C)$ | `scan2d_kernel` with `AXIS=1` |
| **15.3** | `tl.associative_scan()` | $y_i=x_0\oplus\cdots\oplus x_i$ | $\oplus$ associative; $k$ states in, $k$ states out | `assoc_scan_kernel` (`_add`, `_maximum`) |
| **15.4** | Inclusive scan | $y_i=x_0\oplus\cdots\oplus x_i$ | $y_0=x_0$; shape preserved | `inclusive_scan_kernel` |
| **15.5** | Exclusive scan | $y_i=x_0\oplus\cdots\oplus x_{i-1}$ | $y_0=e$; $e_i=y_{i-1}$ | `exclusive_scan_kernel` |
| **15.6** | Parallel scan | $y_i^{(d)}=y_i^{(d/2)}\oplus y_{i-d}^{(d/2)}$ | depth $\lceil\log_2 N\rceil$; shape preserved | `hillis_steele_kernel` |
| **15.7** | Reduction vs scan | $r=\bigoplus_j x_j$ vs $y_i=\bigoplus_{j\le i}x_j$ | reduction removes the axis, scan keeps it | `reduce_scan_kernel` |
| **15.8** | Applications of scan | positions, segments, recurrence | same shape; tuple states for segments/affine | `segmented_scan_kernel`, `compact_kernel`, `recurrence_kernel` |

**One-line thesis.** A reduction answers "what is the total?"; a scan answers "what is the total so far?" at every position, and the answer must fit in the same shape.


## Worked numerical example

The chapter spine is $x=(3,1,4,1,5)$.

### Inclusive and exclusive

| $i$ | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| $x_i$ | 3 | 1 | 4 | 1 | 5 |
| $y_i$ inclusive | 3 | 4 | 8 | 9 | 14 |
| $e_i$ exclusive | 0 | 3 | 4 | 8 | 9 |

- **Recurrence.** $y_i=y_{i-1}+x_i$ with $y_{-1}=0$, and $e_i=y_{i-1}$.
- **Subtract form.** $e_i=y_i-x_i$: $3-3=0$, $4-1=3$, $8-4=4$, $9-1=8$, $14-5=9$.

### Shift-and-scan

- **Shift.** The exclusive scan is the inclusive scan of $(0,3,1,4,1)\mapsto(0,3,4,8,9)$.
- **Identity.** The first zero is the identity, not data.
- **Generality.** This is the route that also works for max, min, and products.

### Hillis–Steele, one doubling per round

| round | $d$ | array |
|---|---:|---|
| 0 | — | 3 1 4 1 5 |
| 1 | 1 | 3 4 5 5 6 |
| 2 | 2 | 3 4 8 9 11 |
| 3 | 4 | 3 4 8 9 14 |

- **Round 1.** $y_1=1+3=4$, $y_2=4+1=5$, $y_3=1+4=5$, $y_4=5+1=6$.
- **Round 2.** $y_2=5+3=8$, $y_3=5+4=9$, $y_4=6+5=11$.
- **Round 3.** $y_4=11+3=14$.
- **Count.** After $\lceil\log_2 5\rceil=3$ rounds the scan is complete. The active combines per round are $4,3,1$, so the work is $8$ adds; a sequential loop uses $4$.

### Why associativity is not optional

- **Left fold.** $(((3-1)-4)-1)-5=-8$.
- **Balanced fold.** $((3-1)-4)-(1-5)=(-2)-(-4)=2$.
- **Warning.** Both are "scans" of a sort; they disagree, so a tree cannot implement a subtraction scan.
- **Floats.** Addition is associative on integers, but reassociation can change the last bits of floats; compare with a tolerance, not bitwise.


In [ ]:
def test_worked_example():
    '''Check every number in the worked example above.'''
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0])
    inclusive = torch.cumsum(x, 0)
    exclusive = inclusive - x
    shifted = torch.cat([torch.zeros(1), x[:-1]])
    exclusive_shift = torch.cumsum(shifted, 0)

    assert torch.equal(inclusive, torch.tensor([3.0, 4.0, 8.0, 9.0, 14.0]))
    assert torch.equal(exclusive, torch.tensor([0.0, 3.0, 4.0, 8.0, 9.0]))
    assert torch.equal(exclusive_shift, exclusive)

    def hillis_steele_reference(values):
        y = values.clone()
        d = 1
        while d < values.numel():
            previous = y.clone()
            y = y + torch.cat([torch.zeros(d, dtype=y.dtype), previous[:-d]])
            d *= 2
        return y

    assert torch.equal(hillis_steele_reference(x), inclusive)

    left = ((x[0] - x[1]) - x[2]) - x[3] - x[4]
    balanced = (x[0] - x[1] - x[2]) - (x[3] - x[4])
    assert left.item() == -8.0
    assert balanced.item() == 2.0
    assert left.item() != balanced.item()

    assert inclusive[-1].item() == x.sum().item() == 14.0
    assert exclusive[-1].item() == x.sum().item() - x[-1].item() == 9.0
    print("worked example: inclusive (3,4,8,9,14), exclusive (0,3,4,8,9), 3 rounds")


## Part II — Method 01: the naive scan loop

- **Connection.** Takes the definition from Part I and writes it as the reference implementation.
- **Rule.** The inclusive loop carries one accumulator from left to right and appends it **after** each add; the exclusive loop appends the accumulator **before** the add.
- **Cost.** Both are $O(N)$ work and $O(N)$ depth: step $i$ waits for step $i-1$. That serial dependency is exactly what the parallel scan removes.
- **Role.** This loop is the correctness oracle. If a Triton kernel disagrees with it on the spine, the kernel is wrong. It is also the right implementation for a short sequence on a scalar CPU; it is the wrong implementation for a million elements on a GPU.


In [ ]:
def naive_inclusive_scan(values):
    '''Method 01: y_i = y_{i-1} + x_i, one Python step per element.'''
    out = []
    accumulator = 0.0
    for value in values:
        accumulator = accumulator + float(value)
        out.append(accumulator)
    return torch.tensor(out, dtype=torch.float32)


def naive_exclusive_scan(values):
    '''Method 01: e_i = y_{i-1}, with e_0 = the identity 0.'''
    out = []
    accumulator = 0.0
    for value in values:
        out.append(accumulator)
        accumulator = accumulator + float(value)
    return torch.tensor(out, dtype=torch.float32)


def test_method01():
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0])
    assert torch.equal(naive_inclusive_scan(x), torch.tensor([3.0, 4.0, 8.0, 9.0, 14.0]))
    assert torch.equal(naive_exclusive_scan(x), torch.tensor([0.0, 3.0, 4.0, 8.0, 9.0]))
    assert torch.equal(naive_inclusive_scan(x), torch.cumsum(x, 0))
    assert torch.equal(naive_exclusive_scan(x), torch.cumsum(x, 0) - x)

    torch.manual_seed(0)
    xr = torch.randn(37)
    torch.testing.assert_close(naive_inclusive_scan(xr), torch.cumsum(xr, 0), atol=1e-6, rtol=1e-6)
    assert torch.allclose(naive_inclusive_scan(xr)[-1], xr.sum(), atol=1e-6, rtol=1e-6)
    print("method 01: the loop matches torch.cumsum on the spine and on n=37")


## Part III — Method 02: Triton scan kernels

- **Connection.** Turns the Part I definitions and the Part II oracle into block kernels.
- **Rule.** All kernels use one program over one block. The block is `BLOCK = next_pow2(n)` and the mask is `offs < n`; a masked lane loads the identity, which leaves every valid prefix unchanged.
- **Primitive availability.** Triton 3.7 provides both `tl.cumsum` and `tl.associative_scan`. They also run under `TRITON_INTERPRET=1`, so the tests execute on CPU in the interpreter and on GPU natively. The manual Hillis–Steele kernel is the readable tree and the fallback for a Triton build without the primitive.
- **Global versus block.** One program can scan one block. For $N\gt $ `BLOCK`, split into blocks, scan each block, then scan the block totals and add the carries; that two-level algorithm is an exercise. Every kernel here uses `BLOCK = next_pow2(N)` so the scan is global for the test sizes. That choice limits the wrappers to sizes that fit one Triton block; the tests stay at $N\le257$. A production kernel fixes `BLOCK` and uses the two-level scheme.
- **Kernel map.**

| Section | Path | Kernel |
|---|---|---|
| 15.1, 15.2, 15.4 | primitive | `inclusive_scan_kernel` (`tl.cumsum`) |
| 15.3 | primitive | `assoc_scan_kernel` (`_add`, `_maximum`) |
| 15.5 | shift + primitive | `exclusive_scan_kernel` |
| 15.6 | manual doubling | `hillis_steele_kernel` |
| 15.7 | reduction + scan | `reduce_scan_kernel` |
| 15.2 (2-D) | primitive, axis | `scan2d_kernel` |


### Inclusive scan kernel

- **Rule.** The kernel is the chapter skeleton with one line changed: load with `other=0.0`, compute `y = tl.cumsum(x, axis=0)`, store under the mask.
- **Identity.** `other=0.0` is the additive identity. A padding lane contributes nothing to any valid prefix, and its own stored value is suppressed by the mask.
- **Test.** The spine plus a random vector of length $257$ forces a partial last block: $\lceil 257/512\rceil=1$, with 255 masked lanes.


In [ ]:
@triton.jit
def inclusive_scan_kernel(x_ptr, y_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.cumsum(x, axis=0)
    tl.store(y_ptr + offs, y, mask=mask)


def inclusive_scan(x):
    x = x.contiguous()
    out = torch.empty_like(x)
    inclusive_scan_kernel[(1,)](x, out, x.numel(), BLOCK=next_pow2(x.numel()))
    return out


def test_inclusive_scan():
    require_gpu("inclusive scan")
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0], device=DEVICE)
    y = inclusive_scan(x)
    torch.testing.assert_close(y, torch.cumsum(x, 0))
    assert torch.equal(y, torch.tensor([3.0, 4.0, 8.0, 9.0, 14.0], device=DEVICE))
    assert y.shape == x.shape

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    yr = inclusive_scan(xr)
    torch.testing.assert_close(yr, torch.cumsum(xr, 0), atol=1e-5, rtol=1e-5)
    assert torch.allclose(yr[-1], xr.sum(), atol=1e-4, rtol=1e-5)
    assert next_pow2(257) - 257 == 255
    print("inclusive scan: spine and n=257 OK")


### Exclusive scan kernel

- **Rule.** Triton has no exclusive flag, so shift the input and scan: `shifted = tl.load(x_ptr + offs - 1, mask=(offs > 0) & mask, other=0.0)` then `exclusive = tl.cumsum(shifted, axis=0)`.
- **Identity.** At `offs == 0` the address is `x_ptr - 1`, but the mask is false, so no memory is touched and `other=0.0` supplies the identity.
- **Contrast.** For comparison the kernel also computes `inclusive - x`, which is valid only because addition is invertible.
- **Generality.** The max case shows why the shift is the general route: there is no `inclusive - x` for a running maximum, but the shift works for every associative operator with an identity.


In [ ]:
@triton.jit
def exclusive_scan_kernel(x_ptr, shift_ptr, sub_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    shifted = tl.load(x_ptr + offs - 1, mask=(offs > 0) & mask, other=0.0)

    inclusive = tl.cumsum(x, axis=0)
    exclusive_shift = tl.cumsum(shifted, axis=0)
    exclusive_sub = inclusive - x

    tl.store(shift_ptr + offs, exclusive_shift, mask=mask)
    tl.store(sub_ptr + offs, exclusive_sub, mask=mask)


def exclusive_scan_both(x):
    x = x.contiguous()
    shift_out = torch.empty_like(x)
    sub_out = torch.empty_like(x)
    exclusive_scan_kernel[(1,)](x, shift_out, sub_out, x.numel(), BLOCK=next_pow2(x.numel()))
    return shift_out, sub_out


def test_exclusive_scan():
    require_gpu("exclusive scan")
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0], device=DEVICE)
    shift_out, sub_out = exclusive_scan_both(x)
    reference = torch.cat([torch.zeros(1, device=DEVICE), torch.cumsum(x, 0)[:-1]])

    torch.testing.assert_close(shift_out, reference)
    torch.testing.assert_close(sub_out, torch.cumsum(x, 0) - x)
    assert torch.equal(shift_out, torch.tensor([0.0, 3.0, 4.0, 8.0, 9.0], device=DEVICE))
    assert shift_out[0].item() == 0.0
    assert shift_out.shape == x.shape

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    shift_r, sub_r = exclusive_scan_both(xr)
    torch.testing.assert_close(shift_r, sub_r, atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(shift_r, torch.cumsum(xr, 0) - xr, atol=1e-5, rtol=1e-5)


### Custom associative scan

- **Rule.** `tl.associative_scan` takes a `@triton.jit` combine function. Addition gives a prefix sum; `tl.maximum` gives a running maximum.
- **Identity.** The max load uses `other=-inf`, the identity for max; using `0.0` would let a padded lane win whenever all real values are negative.
- **Contract.** The combine function must be associative, must accept two scalars (or $2k$ scalars for $k$ state tensors), and must return the same number of values.
- **Kernel.** One launch runs both scans and stores both prefix buffers.


In [ ]:
@triton.jit
def _add(a, b):
    return a + b


@triton.jit
def _maximum(a, b):
    return tl.maximum(a, b)


@triton.jit
def assoc_scan_kernel(x_ptr, sum_ptr, max_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    x_for_max = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))

    prefix_sum = tl.associative_scan(x, axis=0, combine_fn=_add)
    prefix_max = tl.associative_scan(x_for_max, axis=0, combine_fn=_maximum)

    tl.store(sum_ptr + offs, prefix_sum, mask=mask)
    tl.store(max_ptr + offs, prefix_max, mask=mask)


def assoc_scan(x):
    x = x.contiguous()
    sums = torch.empty_like(x)
    maxima = torch.empty_like(x)
    assoc_scan_kernel[(1,)](x, sums, maxima, x.numel(), BLOCK=next_pow2(x.numel()))
    return sums, maxima


def test_assoc_scan():
    require_gpu("associative scan")
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0], device=DEVICE)
    sums, maxima = assoc_scan(x)
    torch.testing.assert_close(sums, torch.cumsum(x, 0))
    torch.testing.assert_close(maxima, torch.cummax(x, 0).values)
    assert torch.equal(maxima, torch.tensor([3.0, 3.0, 4.0, 4.0, 5.0], device=DEVICE))

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    sums_r, maxima_r = assoc_scan(xr)
    torch.testing.assert_close(sums_r, torch.cumsum(xr, 0), atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(maxima_r, torch.cummax(xr, 0).values)
    print("associative scan: add and max match torch")


### Manual Hillis–Steele kernel

- **Rule.** `step` starts at 1 and doubles; `tl.gather` supplies lane `i - step`, and `tl.where(offs >= step, shifted, 0.0)` suppresses the read for the first `step` lanes.
- **Schedule.** Because `tl.static_range` unrolls at compile time and `step` is a Python integer, every round is a constant-distance shuffle.
- **Rounds.** `LOG_BLOCK = log2(BLOCK)`: the spine uses `BLOCK=8` and three rounds; $n=257$ uses `BLOCK=512` and nine rounds.
- **Test.** The result must equal `tl.cumsum`: the manual tree and the primitive agree.


In [ ]:
@triton.jit
def hillis_steele_kernel(x_ptr, y_ptr, n_elements, BLOCK: tl.constexpr,
                         LOG_BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    acc = tl.load(x_ptr + offs, mask=mask, other=0.0)

    step = 1
    for _ in tl.static_range(LOG_BLOCK):
        index = tl.maximum(offs - step, 0)
        shifted = tl.gather(acc, index, axis=0)
        acc = acc + tl.where(offs >= step, shifted, 0.0)
        step = step * 2

    tl.store(y_ptr + offs, acc, mask=mask)


def hillis_steele(x):
    x = x.contiguous()
    out = torch.empty_like(x)
    block = next_pow2(x.numel())
    hillis_steele_kernel[(1,)](
        x, out, x.numel(), BLOCK=block, LOG_BLOCK=int(math.log2(block))
    )
    return out


def test_hillis_steele():
    require_gpu("Hillis-Steele scan")
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0], device=DEVICE)
    y = hillis_steele(x)
    torch.testing.assert_close(y, torch.cumsum(x, 0))
    assert torch.equal(y, torch.tensor([3.0, 4.0, 8.0, 9.0, 14.0], device=DEVICE))
    assert math.ceil(math.log2(5)) == 3
    assert next_pow2(5) == 8

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    yr = hillis_steele(xr)
    torch.testing.assert_close(yr, torch.cumsum(xr, 0), atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(yr, inclusive_scan(xr), atol=1e-5, rtol=1e-5)


### 2-D scan along a chosen axis

- **Rule.** A 2-D tile is loaded with row and column offsets, $\text{address}_{r,c}=r\cdot C+c$, and the scan runs along one axis of the block.
- **Axis.** `AXIS` is a `tl.constexpr`, so the branch disappears at compile time. The output shape is still $(R,C)$.
- **Example.** For $A=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$, `axis=1` gives $\begin{bmatrix}1&3&6\\4&9&15\end{bmatrix}$ and `axis=0` gives $\begin{bmatrix}1&2&3\\5&7&9\end{bmatrix}$.
- **Test.** They agree only at the top-left entry; the test asserts both.


In [ ]:
@triton.jit
def scan2d_kernel(x_ptr, y_ptr, rows, cols, BLOCK_R: tl.constexpr,
                  BLOCK_C: tl.constexpr, AXIS: tl.constexpr):
    r = tl.arange(0, BLOCK_R)[:, None]
    c = tl.arange(0, BLOCK_C)[None, :]
    mask = (r < rows) & (c < cols)

    x = tl.load(x_ptr + r * cols + c, mask=mask, other=0.0)
    if AXIS == 0:
        y = tl.cumsum(x, axis=0)
    else:
        y = tl.cumsum(x, axis=1)

    tl.store(y_ptr + r * cols + c, y, mask=mask)


def scan2d(x, axis):
    assert x.ndim == 2
    assert axis in (0, 1)
    x = x.contiguous()
    rows, cols = x.shape
    out = torch.empty_like(x)
    scan2d_kernel[(1,)](
        x, out, rows, cols,
        BLOCK_R=next_pow2(rows), BLOCK_C=next_pow2(cols), AXIS=axis,
    )
    return out


def test_scan2d():
    require_gpu("2-D scan")
    a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=DEVICE)
    row_scan = scan2d(a, axis=1)
    col_scan = scan2d(a, axis=0)

    torch.testing.assert_close(row_scan, torch.cumsum(a, 1))
    torch.testing.assert_close(col_scan, torch.cumsum(a, 0))
    assert row_scan.shape == a.shape
    assert col_scan.shape == a.shape
    assert torch.sum(a, 0).shape == (3,)
    assert torch.sum(a, 1).shape == (2,)
    assert not torch.equal(row_scan, col_scan)

    torch.manual_seed(0)
    big = torch.randn(7, 5, device=DEVICE)
    torch.testing.assert_close(scan2d(big, axis=0), torch.cumsum(big, 0), atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(scan2d(big, axis=1), torch.cumsum(big, 1), atol=1e-5, rtol=1e-5)
    print("2-D scan: axis=0 and axis=1 both match torch.cumsum")


### Reduction vs scan in one kernel

- **Rule.** The kernel stores the prefix tile and a one-element total, making the shape rule concrete.
- **Shapes.** The prefix buffer has the input shape $(N,)$; the total buffer has shape $()$, one element; the invariant `prefix[-1] == total` ties them together.
- **Choice.** If a downstream computation only needs the total, `tl.sum` is enough. If it needs every offset, the scan is required. Producing both costs one extra reduction, not a second pass over the input.


In [ ]:
@triton.jit
def reduce_scan_kernel(x_ptr, prefix_ptr, total_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    prefix = tl.cumsum(x, axis=0)
    total = tl.sum(x, axis=0)

    tl.store(prefix_ptr + offs, prefix, mask=mask)
    tl.store(total_ptr, total)


def reduce_scan(x):
    x = x.contiguous()
    prefix = torch.empty_like(x)
    total = torch.empty(1, dtype=x.dtype, device=x.device)
    reduce_scan_kernel[(1,)](x, prefix, total, x.numel(), BLOCK=next_pow2(x.numel()))
    return prefix, total


def test_reduce_scan():
    require_gpu("reduction vs scan")
    x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0], device=DEVICE)
    prefix, total = reduce_scan(x)

    assert prefix.shape == x.shape
    assert total.numel() == 1
    torch.testing.assert_close(prefix, torch.cumsum(x, 0))
    torch.testing.assert_close(total[0], x.sum())
    assert torch.allclose(prefix[-1], total[0])
    assert total[0].item() == 14.0

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    prefix_r, total_r = reduce_scan(xr)
    torch.testing.assert_close(prefix_r[-1], total_r[0], atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(total_r[0], xr.sum(), atol=1e-4, rtol=1e-5)


## Part IV — Method 03: applied scans

- **Connection.** Composes the Part III kernels into the 15.8 patterns.
- **Applications.**
  1. **Segmented scan:** per-segment position ids and per-segment sums.
  2. **Stream compaction:** the exclusive scan of a keep mask gives write offsets.
  3. **First-order recurrence:** an affine tuple scan evaluates $y_i=a_i y_{i-1}+b_i$.
- **Shape.** All three preserve the input shape for the scan itself. Compaction changes only where the kept values are stored.


### Application 1 — segmented positions and segmented sums

- **Connection.** Builds on 15.8's segmented-sum pattern and 15.3's tuple scan.
- **Rule.** A plain cumsum over concatenated sequences counts across boundaries. Add a **head flag** $h_i\in\{0,1\}$ that is 1 at the first element of each segment, and combine $(h_1,v_1)\oplus(h_2,v_2)=(h_1\lor h_2,\ \operatorname{where}(h_2,\ v_2,\ v_1+v_2))$.
- **Why it works.** If the right operand starts a segment, its value wins; otherwise the left prefix is added. The head flag is an absorbing "restart" marker, so the operator is associative.
- **Example.** For $h=(1,0,0,1,0,0)$ and $x=(1,2,3,4,5,6)$, the segmented sums are $(1,3,6,\ 4,9,15)$. Scanning ones with the same operator gives $(1,2,3,\ 1,2,3)$, so subtracting one gives the positions $(0,1,2,\ 0,1,2)$.
- **Triton.** The kernel carries the tuple `(head, value)` through `tl.associative_scan`: two states in, two states out.


In [ ]:
@triton.jit
def _segmented_combine(h1, v1, h2, v2):
    return h1 | h2, tl.where(h2, v2, v1 + v2)


@triton.jit
def segmented_scan_kernel(head_ptr, value_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements

    head = tl.load(head_ptr + offs, mask=mask, other=0) != 0
    value = tl.load(value_ptr + offs, mask=mask, other=0.0)

    _, scanned = tl.associative_scan((head, value), axis=0, combine_fn=_segmented_combine)
    tl.store(out_ptr + offs, scanned, mask=mask)


def segmented_scan(heads, values):
    heads = heads.contiguous()
    values = values.contiguous()
    out = torch.empty_like(values)
    segmented_scan_kernel[(1,)](
        heads.to(torch.int32), values, out, values.numel(), BLOCK=next_pow2(values.numel())
    )
    return out


def test_segmented_scan():
    require_gpu("segmented scan")
    heads = torch.tensor([1, 0, 0, 1, 0, 0], dtype=torch.int32, device=DEVICE)
    values = torch.arange(1, 7, dtype=torch.float32, device=DEVICE)

    sums = segmented_scan(heads, values)
    torch.testing.assert_close(sums, torch.tensor([1.0, 3.0, 6.0, 4.0, 9.0, 15.0], device=DEVICE))

    ones = torch.ones(6, device=DEVICE)
    positions = segmented_scan(heads, ones) - 1
    torch.testing.assert_close(positions, torch.tensor([0.0, 1.0, 2.0, 0.0, 1.0, 2.0], device=DEVICE))

    torch.manual_seed(0)
    n = 37
    values_r = torch.randn(n, device=DEVICE)
    heads_r = torch.zeros(n, dtype=torch.int32, device=DEVICE)
    starts = torch.tensor([0, 11, 12, 25], device=DEVICE)
    heads_r[starts] = 1

    reference = []
    accumulator = 0.0
    for value, head in zip(values_r.tolist(), heads_r.tolist()):
        accumulator = value if head else accumulator + value
        reference.append(accumulator)
    segmented = segmented_scan(heads_r, values_r)
    torch.testing.assert_close(segmented, torch.tensor(reference, device=DEVICE),
                               atol=1e-5, rtol=1e-5)
    print("segmented scan: positions (0,1,2,0,1,2) and segment sums OK")


### Application 2 — stream compaction

- **Connection.** Builds on 15.5's exclusive scan and 15.8's compaction pattern.
- **Rule.** Keep only the elements selected by a mask and pack them to the front. The destination of item $i$ is the number of kept items before it: $\operatorname{pos}_i=\sum_{j\lt i}k_j$, the exclusive scan of the keep mask $k$.
- **Scatter.** `keep = tl.load(keep_ptr + offs, mask=mask, other=0) != 0`; `pos = tl.cumsum(keep.to(tl.int32), axis=0) - keep.to(tl.int32)`; `tl.store(out_ptr + pos, x, mask=mask & keep)`.
- **Shape.** The scan still has length $N$; only the **output tensor** shrinks to $K=\sum_j k_j$.
- **Cost.** The wrapper sizes the output with `keep.sum().item()`, a host-side reduction that synchronizes; a production version would allocate the upper bound and write the count on the device.
- **Stability.** The order is stable because the offsets increase with the input index. A permutation of equal keys would still compact, but it would not preserve their relative order.


In [ ]:
@triton.jit
def compact_kernel(x_ptr, keep_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements

    keep = tl.load(keep_ptr + offs, mask=mask, other=0) != 0
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)

    keep_i32 = keep.to(tl.int32)
    position = tl.cumsum(keep_i32, axis=0) - keep_i32
    tl.store(out_ptr + position, x, mask=mask & keep)


def compact(x, keep):
    x = x.contiguous()
    keep = keep.contiguous().to(torch.int32)
    out = torch.empty(int(keep.sum().item()), dtype=x.dtype, device=x.device)
    if out.numel() > 0:
        compact_kernel[(1,)](x, keep, out, x.numel(), BLOCK=next_pow2(x.numel()))
    return out


def test_compact():
    require_gpu("stream compaction")
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device=DEVICE)
    keep = torch.tensor([1, 0, 1, 0, 1, 1], dtype=torch.int32, device=DEVICE)
    out = compact(x, keep)
    torch.testing.assert_close(out, torch.tensor([1.0, 3.0, 5.0, 6.0], device=DEVICE))
    assert out.shape == (4,)

    torch.manual_seed(0)
    xr = torch.randn(257, device=DEVICE)
    keep_r = (torch.rand(257, device=DEVICE) > 0.5).to(torch.int32)
    torch.testing.assert_close(compact(xr, keep_r), xr[keep_r.bool()])

    empty = compact(x, torch.zeros(6, dtype=torch.int32, device=DEVICE))
    assert empty.numel() == 0


### Application 3 — a first-order recurrence

- **Connection.** Builds on 15.8's affine pattern and 15.3's tuple scan.
- **Rule.** A first-order linear recurrence is $y_i=a_i\,y_{i-1}+b_i$ with $y_{-1}=0$.
- **Composition.** Each step is an affine map $T_i(y)=a_i y+b_i$, and $T_2(T_1(y))=a_2(a_1y+b_1)+b_2=(a_1a_2)y+(a_2b_1+b_2)$.
- **Monoid.** The pair $(a,b)$ combines as $(a_1,b_1)\oplus(a_2,b_2)=(a_1a_2,\ a_2b_1+b_2)$, with identity $(1,0)$.
- **Example.** For $a=(1,2,3,4,5)$ and $b=(1,1,1,1,1)$, $y=(1,\ 3,\ 10,\ 41,\ 206)$. Lane 0 is $1$, lane 1 is $2\cdot1+1=3$, lane 2 is $3\cdot3+1=10$, lane 3 is $4\cdot10+1=41$, lane 4 is $5\cdot41+1=206$.
- **Why.** The scan evaluates all five steps with logarithmic depth, even though the recurrence is inherently sequential.


In [ ]:
@triton.jit
def _affine_combine(a1, b1, a2, b2):
    return a1 * a2, a2 * b1 + b2


@triton.jit
def recurrence_kernel(a_ptr, b_ptr, y_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements

    a = tl.load(a_ptr + offs, mask=mask, other=1.0)
    b = tl.load(b_ptr + offs, mask=mask, other=0.0)

    _, y = tl.associative_scan((a, b), axis=0, combine_fn=_affine_combine)
    tl.store(y_ptr + offs, y, mask=mask)


def first_order_recurrence(a, b):
    a = a.contiguous()
    b = b.contiguous()
    out = torch.empty_like(b)
    recurrence_kernel[(1,)](a, b, out, b.numel(), BLOCK=next_pow2(b.numel()))
    return out


def test_recurrence():
    require_gpu("first-order recurrence")
    a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], device=DEVICE)
    b = torch.ones(5, device=DEVICE)
    y = first_order_recurrence(a, b)
    torch.testing.assert_close(y, torch.tensor([1.0, 3.0, 10.0, 41.0, 206.0], device=DEVICE))

    torch.manual_seed(0)
    n = 37
    a_r = torch.rand(n, device=DEVICE) * 0.5 + 0.5
    b_r = torch.randn(n, device=DEVICE)
    y_r = first_order_recurrence(a_r, b_r)

    reference = []
    previous = 0.0
    for ai, bi in zip(a_r.tolist(), b_r.tolist()):
        previous = ai * previous + bi
        reference.append(previous)
    torch.testing.assert_close(y_r, torch.tensor(reference, device=DEVICE), atol=1e-5, rtol=1e-5)
    print("first-order recurrence: (1,3,10,41,206) and n=37 OK")


## Work and depth

- **Sequential loop.** $W_{\text{seq}}=N-1$ and $D_{\text{seq}}=N-1$.
- **Hillis–Steele.** With $m=\lceil\log_2 N\rceil$ and rounds $d=1,2,4,\dots$: $D_{\text{HS}}=m$ and $W_{\text{HS}}=\sum_{k=0}^{m-1}(N-2^k)=mN-(2^m-1)$. Counting every lane, including the suppressed ones, gives the simpler bound $W_{\text{HS}}\le mN$.
- **Blelloch (work-efficient).** $W_{\text{Blelloch}}=2(N-1)$ and $D_{\text{Blelloch}}=2\lceil\log_2 N\rceil$.

| $N$ | loop $W/D$ | Hillis–Steele $W/D$ | Blelloch $W/D$ |
|---:|---:|---:|---:|
| 5 | 4 / 4 | 8 / 3 | 8 / 6 |
| 8 | 7 / 7 | 17 / 3 | 14 / 6 |
| 1024 | 1023 / 1023 | 9217 / 10 | 2046 / 20 |

- **Parallelism.** The parallel algorithms do more work but have logarithmic depth. The parallelism is $W/D$: Hillis–Steele exposes about $N$ independent operations, Blelloch about $N/2$. A machine with $P$ lanes cannot beat $S(P)\le\min(P,W/D)$.
- **Memory.** A scan reads and writes each element once: $Q\approx2Nb$ bytes for $N$ elements of $b$ bytes. At large $N$ the kernel is bandwidth-bound and the tree's extra adds hide behind memory traffic; at small $N$ the shuffle and layout-conversion overhead dominates.
- **Segmented scan.** A block-wide segmented Hillis–Steele has the same depth as an unsegmented one, $\lceil\log_2 N\rceil$, because the tree still spans all $N$ lanes; the head flag changes the values, not the schedule. Processing each segment in its own program can reduce the depth to $\lceil\log_2 L\rceil$ for the longest segment of length $L$, at the cost of a segmented launch and a per-segment offset table. The tuple doubles the state traffic and the combine cost, but not the depth.


In [ ]:
def sequential_work(n):
    return max(n - 1, 0)


def sequential_depth(n):
    return max(n - 1, 0)


def hs_rounds(n):
    return 0 if n <= 1 else math.ceil(math.log2(n))


def hs_depth(n):
    return hs_rounds(n)


def hs_work(n):
    work = 0
    distance = 1
    while distance < n:
        work += n - distance
        distance *= 2
    return work


def blelloch_work(n):
    return 2 * (n - 1)


def blelloch_depth(n):
    return 2 * hs_rounds(n)


def test_work_depth():
    assert (sequential_work(5), sequential_depth(5)) == (4, 4)
    assert (hs_work(5), hs_depth(5)) == (8, 3)
    assert (blelloch_work(5), blelloch_depth(5)) == (8, 6)

    assert (sequential_work(8), sequential_depth(8)) == (7, 7)
    assert (hs_work(8), hs_depth(8)) == (17, 3)
    assert (blelloch_work(8), blelloch_depth(8)) == (14, 6)

    assert hs_work(1024) == 1024 * 10 - 1023 == 9217
    assert hs_depth(1024) == 10
    assert blelloch_work(1024) == 2046
    assert blelloch_depth(1024) == 20

    assert hs_work(8) > sequential_work(8)
    assert hs_depth(8) < sequential_depth(8)
    assert blelloch_work(8) < hs_work(8)
    assert blelloch_depth(8) > hs_depth(8)
    print("work/depth: n=8 -> loop 7/7, Hillis-Steele 17/3, Blelloch 14/6")


## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| inclusive vs exclusive | every offset is one element late or early | state the convention; use $e_i=y_{i-1}$ or `inclusive - x` |
| wrong axis | a row-wise scan becomes a column-wise scan | pass `axis=1` for rows; assert the output shape |
| non-associative combine | tree order and loop order disagree | use $+,\times,\min,\max$ or prove associativity |
| float order | last-bit differences from `torch.cumsum` | compare with `assert_close(atol=1e-5)`, not `equal` |
| masked lanes | a padding value wins a max or pollutes a sum | `other` = identity: `0.0`, `1.0`, `-inf`, `+inf` |
| identity element | exclusive slot 0 contains garbage | shift with `other=e`; never leave the first lane unmasked |
| overflow | `int32`/`float16` prefix sums wrap or saturate | accumulate in `int64`/`float32` |
| block boundary | each block restarts at zero | scan block totals and add the carries |
| one program per row | a 2-D scan compiles but scans the wrong axis | check `AXIS` and the offsets |
| scan for a total | needless work and memory | use `tl.sum` when only the last value is used |
| compaction order | equal keys appear permuted | keep the exclusive offsets in input order |

- **Highest-cost mistakes.** The inclusive/exclusive off-by-one and the wrong axis. Both are shape-correct, so no assertion fires unless you write the explicit value check.


## Self-check

1. For $x=(3,1,4,1,5)$, write the inclusive and exclusive scans and the shift that converts one into the other.
2. Why does `tl.cumsum(x, axis=0)` preserve the shape while `tl.sum(x, axis=0)` does not?
3. Give a combine function that is commutative but not associative, and show two different scans of the same input.
4. Why is `inclusive - x` a valid exclusive scan for addition but not for maximum?
5. How many rounds does Hillis–Steele need for $N=1000$? How many adds does it perform?
6. If 32 segments of length 32 are scanned in separate programs, what is the depth? What is the depth of one block-wide segmented scan over all 1024 elements?
7. In stream compaction, why is the scan length $N$ while the output length is $K$?
8. Derive the affine combine $(a_1,b_1)\oplus(a_2,b_2)$ from $T_i(y)=a_i y+b_i$.


## Summary

- A scan is a reduction that keeps every prefix; its output shape equals its input shape.
- Inclusive keeps the current element; exclusive drops it and starts at the identity.
- `tl.cumsum` is the sum scan; `tl.associative_scan` accepts any associative `@triton.jit` combine function.
- The exclusive scan is a shifted inclusive scan; the subtract shortcut works only for invertible operators.
- Hillis–Steele has depth $\lceil\log_2 N\rceil$ and work $\approx N\log_2 N$; Blelloch has work $2(N-1)$ and depth $2\lceil\log_2 N\rceil$.
- Scans produce offsets: position ids, compaction slots, segment boundaries, and recurrence solutions.
- The scan is memory-bound at large $N$; the extra tree work is usually hidden, but the shuffles are not free.

**One sentence.** A reduction collapses a sequence; a scan preserves it and replaces each element by the combination of its prefix.

**Where this leads.** Chapter 16 composes reductions and scans into stable softmax, log-sum-exp, LayerNorm statistics, and online softmax.


## Exercises

1. **Exclusive max.** Write an exclusive running maximum with the shift-and-scan route, and explain why `inclusive - x` fails.
2. **Chunked global scan.** Split a vector of length $N=1024$ into blocks of 128, scan each block, scan the 8 block totals, and add the carries. Verify against `torch.cumsum`.
3. **Suffix scan.** Use `reverse=True` in `tl.associative_scan` to compute suffix sums, and check that `suffix[0]` is the total.
4. **Product scan.** Implement a cumulative product with identity `1.0` and positive inputs; compare with `torch.cumprod`.
5. **Running argmax.** Carry the pair `(value, index)` through a tuple scan; break ties by the smaller index.
6. **Compaction stability.** Run the compaction kernel on keys with duplicates and show that the output order follows the input order.
7. **Nonzero initial state.** Modify the recurrence kernel for $y_{-1}=5$; the answer is `B_i + A_i * 5`.
8. **Work count.** For $N=1000$, compute the Hillis–Steele and Blelloch work and depth, and the parallelism of each.
9. **Float reassociation.** Find three `float32` values whose left-to-right sum differs from a balanced sum; explain which kernel order you expect.
10. **Segment mean.** Extend the segmented scan to carry `(sum, count)` and produce per-segment means in one pass.


In [ ]:
CPU_TESTS = [
    test_worked_example,
    test_method01,
    test_work_depth,
]

GPU_TESTS = [
    test_inclusive_scan,
    test_exclusive_scan,
    test_assoc_scan,
    test_hillis_steele,
    test_scan2d,
    test_reduce_scan,
    test_segmented_scan,
    test_compact,
    test_recurrence,
]

print("CPU tests")
for test in CPU_TESTS:
    test()

print()
print("GPU tests")
for test in GPU_TESTS:
    run_if_gpu(test)

print()
print("done: CPU checks passed; GPU checks ran or printed SKIP")
